# AQUALERT — Modelagem Preditiva de NDCI com Prophet + Assimilação AR(1)
### Estimativa e Previsão de Floração de Algas a partir do Sentinel-2 (270 Pixels Únicos de 20m)
Este notebook implementa o pipeline científico auditado (Frente 1 + Frente 2):
1. **Amostragem Integral de Grade Nativa (`GEO-05`)**: 270 pixels únicos de 20m (`EPSG:32723`) em buffer de 175m (350m de diâmetro) centrado em `-22.530116459790744, -44.5639148196765`.
2. **Controle de Qualidade Óptica e Física (`GEO-02`, `GEO-03`, `GEO-06`, `ML-01`)**: `QA60`, `MSK_CLDPRB < 20%`, `SCL {2,3,8,9,10,11}`, `NDWI > 0.05` e `valid_pixel_count >= 81` (`>= 30%`), **sem poda estatística circular pelo Prophet** (preservando 100% das florações algais reais).
3. **Seleção de Hiperparâmetros Estritamente no Treino (`ML-02`, `ML-07`)**: Avaliação causal via *Rolling Cross-Validation* ancorada em datas reais de passagem do Sentinel-2 (`61 cutoffs` no conjunto de treino, zero vazamento do holdout).
4. **Assimilação Causal de Estado AR(1) (`ML-04`, `ML-06`)**: Ancoragem na última observação óptica válida com relaxação exponencial ($\tau = 6.5\text{ dias}$) para a trajetória sazonal do Prophet.
5. **Validação Multijanela (1 a 45 dias) e Calibração Empírica de Incerteza (`ML-03`, `ML-05`, `ML-08`)**: Comparação contra *Persistência* e *Climatologia Sazonal* em 8 faixas de horizonte (`3, 5, 7, 10, 15, 21, 30 e 45 dias`).

In [1]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics
from prophet.serialize import model_to_json
from sklearn.metrics import mean_squared_error, mean_absolute_error

from treinamento_ndci_prophet import (
    apply_causal_optical_qa,
    get_observation_anchored_cutoffs,
    compute_causal_baselines_and_assimilation,
    build_operational_horizon_tables,
    evaluate_alert_classification,
    DEFAULT_TAU_DAYS,
)

## 1. Carga dos Dados Extraídos do Sentinel-2 e Controle de Qualidade Óptica (`ML-01`)
A série contém 519 datas únicas observadas (`2019-01-02` a `2026-10-04`), com média espacial sobre até 270 pixels únicos de 20m (`y`), desvio padrão (`std_dev`) e contagem de pixels válidos (`points_count`). Nenhuma floração real é removida por filtro estatístico circular.

In [2]:
df_raw = pd.read_csv('dados_ndci_funil.csv')
df_clean, qa_report = apply_causal_optical_qa(df_raw)

print(f"Total bruto: {qa_report['total_bruto']} | Aprovado pelo QA óptico: {qa_report['total_aprovado_qa']} | Rejeitados: {qa_report['total_bruto'] - qa_report['total_aprovado_qa']}")
print(f"Período: {df_clean['ds'].min().strftime('%Y-%m-%d')} a {df_clean['ds'].max().strftime('%Y-%m-%d')}")
print(f"Eventos de Atenção (NDCI >= 0.12): {qa_report['eventos_atencao_gte_012']} | Eventos Críticos (NDCI >= 0.20): {qa_report['eventos_criticos_gte_020']}")
df_clean.head(10)

Total bruto: 519 | Aprovado pelo QA óptico: 519 | Rejeitados: 0
Período: 2019-01-02 a 2026-10-04
Eventos de Atenção (NDCI >= 0.12): 82 | Eventos Críticos (NDCI >= 0.20): 49


## 2. Divisão Temporal Estrita (80% Treino / 20% Holdout) e Seleção Causal de Hiperparâmetros (`ML-02`, `ML-07`)
A seleção de hiperparâmetros ocorre exclusivamente sobre os `61 cutoffs` observacionais contidos no conjunto de treino (`415` observações até `2025-05-17`), sem jamais consultar as `104` observações de holdout (`2025-05-19` a `2026-10-04`).

In [3]:
train_size = int(len(df_clean) * 0.80)
df_train = df_clean.iloc[:train_size].copy().reset_index(drop=True)
df_test = df_clean.iloc[train_size:].copy().reset_index(drop=True)

with open('relatorio_metricas.json', 'r', encoding='utf-8') as f:
    report_saved = json.load(f)
best_config = report_saved['melhor_configuracao']
print(f"Treino: {len(df_train)} obs ({df_train['ds'].min():%Y-%m-%d} a {df_train['ds'].max():%Y-%m-%d}) | Holdout: {len(df_test)} obs ({df_test['ds'].min():%Y-%m-%d} a {df_test['ds'].max():%Y-%m-%d})")
print(f"Melhor configuração selecionada no treino: {best_config['configuracao']} (RMSE={best_config['RMSE']:.4f}, MAE={best_config['MAE']:.4f})")
pd.DataFrame(report_saved['comparacao_teste'])

Treino: 415 obs (2019-01-02 a 2025-05-17) | Holdout: 104 obs (2025-05-19 a 2026-10-04)
Melhor configuração selecionada no treino: Sazonal Regularizada (yearly=8, cp=0.03, sp=1.0, sem semanal) (RMSE=0.1112, MAE=0.0852)


## 3. Validação Cruzada Observacional em Múltiplas Janelas (1 a 45 dias) e Baselines (`ML-03`, `ML-04`, `ML-08`)
Avaliação em `81 cutoffs` reais de passagem do Sentinel-2 para evitar ressonância orbital nos múltiplos de 5 dias e popular todas as faixas de horizonte (`1-3d`, `4-5d`, `6-7d`, `8-10d`, `11-15d`, `16-21d`, `22-30d`, `31-45d`).

In [4]:
m_cv = Prophet(
    yearly_seasonality=int(best_config['yearly_seasonality']),
    weekly_seasonality=bool(best_config['weekly_seasonality']),
    daily_seasonality=False,
    changepoint_prior_scale=float(best_config['changepoint_prior_scale']),
    seasonality_prior_scale=float(best_config['seasonality_prior_scale']),
    interval_width=0.80,
)
m_cv.fit(df_clean[['ds', 'y']])

cutoffs_all = get_observation_anchored_cutoffs(df_clean, initial_days=730, min_step_days=21, horizon_days=45)
df_cv = cross_validation(m_cv, cutoffs=cutoffs_all, horizon='45 days')
df_p = performance_metrics(df_cv, metrics=['mse', 'rmse', 'mae'])

df_cv_full = compute_causal_baselines_and_assimilation(df_cv, df_clean, tau_days=DEFAULT_TAU_DAYS)
metricas_cv_compat, metricas_operacionais = build_operational_horizon_tables(df_cv_full)
df_op = pd.DataFrame(metricas_operacionais)
print(df_op[['horizonte_alvo', 'faixa_dias', 'n_pares', 'RMSE', 'MAE', 'RMSE_prophet_puro', 'RMSE_persistencia', 'RMSE_climatologia', 'skill_vs_persistencia_pct', 'q80_abs_err']].to_string(index=False))

horizonte_alvo faixa_dias  n_pares   RMSE    MAE  RMSE_prophet_puro  RMSE_persistencia  RMSE_climatologia  skill_vs_persistencia_pct  q80_abs_err
        3 dias   1-3 dias       37 0.0912 0.0697             0.0952             0.1050             0.0990                      13.13       0.1048
        5 dias   4-5 dias       37 0.0939 0.0706             0.1004             0.1064             0.0956                      11.69       0.1079
        7 dias   6-7 dias       24 0.1168 0.0929             0.1191             0.1530             0.1209                      23.68       0.1456
       10 dias  8-10 dias       55 0.1069 0.0839             0.1228             0.1065             0.1248                      -0.34       0.1176
       15 dias 11-15 dias       75 0.1153 0.0849             0.1172             0.1401             0.1156                      17.69       0.1305
       21 dias 16-21 dias       83 0.1139 0.0856             0.1164             0.1232             0.1132                   

## 4. Avaliação no Holdout Independente e Detecção de Eventos de Floração (`NDCI >= 0.12`)
Verificação de desempenho nos 20% finais da série (`2025-05-19` a `2026-10-04`) e métricas de classificação para triagem preventiva de florações algais.

In [5]:
holdout_eval = report_saved['avaliacao_holdout_independente']
bloom_eval = evaluate_alert_classification(df_cv_full, threshold=0.12)

r15 = holdout_eval['rolling_operacional_1_15d']
print(f"Holdout Rolling (1-15d): RMSE Assimilado={r15['RMSE_assimilado']:.4f} | MAE={r15['MAE_assimilado']:.4f} | RMSE Persistência={r15['RMSE_persistencia']:.4f} | RMSE Climatologia={r15['RMSE_climatologia']:.4f}")
print(f"Alertas (NDCI >= 0.12, 1-15d): Precisão={bloom_eval['precisao']*100:.1f}% | Sensibilidade={bloom_eval['sensibilidade_recall']*100:.1f}% | Taxa de Falso Alarme={bloom_eval['taxa_falso_alarme']*100:.1f}%")

Holdout Rolling (1-15d): RMSE Assimilado=0.1097 | MAE=0.0847 | RMSE Persistência=0.1490 | RMSE Climatologia=0.1156
Alertas (NDCI >= 0.12, 1-15d): Precisão=53.1% | Sensibilidade=45.9% | Taxa de Falso Alarme=7.9%
